# 03 — Candidate Profiling
## Tamil Nadu Assembly Elections (1971–2021)

**Objectives:**
- Gender analysis (trends, win rates, party-wise)
- Education & profession vs win rate
- Incumbency win rates over time
- Turncoat frequency & success rates
- Career longevity (terms, contests, party loyalty)

In [ ]:
import sys
sys.path.insert(0, '/app')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from db import query, query_all, GENERAL_ELECTION_YEARS, MAJOR_PARTIES

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

df = query_all()
ge = df[df.year.isin(GENERAL_ELECTION_YEARS)].copy()
ge['is_winner'] = ge.position == 1
recent = ge[ge.year >= 2011].copy()
print(f'Loaded {len(ge):,} general election records, {len(recent):,} from 2011+')

## 1. Gender Analysis

In [ ]:
# Win rate by gender over time
gender_win = ge.groupby(['year', 'sex']).agg(
    total=('candidate', 'count'),
    wins=('is_winner', 'sum')
).reset_index()
gender_win['win_rate'] = gender_win.wins / gender_win.total * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Win rate comparison
for sex, color in [('M', '#2196F3'), ('F', '#E91E63')]:
    d = gender_win[gender_win.sex == sex]
    ax1.plot(d.year, d.win_rate, 'o-', label=sex, color=color, linewidth=2)
ax1.set_title('Win Rate by Gender Over Time')
ax1.set_xlabel('Election Year')
ax1.set_ylabel('Win Rate (%)')
ax1.legend(title='Sex')
ax1.grid(True, alpha=0.3)

# Female winners count
female_winners = ge[(ge.sex == 'F') & (ge.is_winner)].groupby('year').size()
ax2.bar(female_winners.index, female_winners.values, color='#E91E63', edgecolor='white')
ax2.set_title('Number of Female Winners per Election')
ax2.set_xlabel('Election Year')
ax2.set_ylabel('Female Winners')
for yr, cnt in zip(female_winners.index, female_winners.values):
    ax2.annotate(str(cnt), (yr, cnt), textcoords='offset points', xytext=(0, 5), ha='center')

plt.tight_layout()
plt.savefig('/app/output/03_gender_win_rates.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Party-wise female candidate percentage (recent elections)
party_gender = recent[recent.party.isin(MAJOR_PARTIES)].groupby(['party', 'sex']).size().unstack(fill_value=0)
if 'F' in party_gender.columns and 'M' in party_gender.columns:
    party_gender['female_pct'] = party_gender['F'] / (party_gender['F'] + party_gender['M']) * 100
    party_gender = party_gender.sort_values('female_pct', ascending=True)

    fig, ax = plt.subplots(figsize=(12, 6))
    party_gender['female_pct'].plot(kind='barh', ax=ax, color='#E91E63', edgecolor='white')
    ax.set_title('Female Candidate % by Major Party (2011–2021)')
    ax.set_xlabel('Female Candidates (%)')
    ax.set_ylabel('')
    for i, v in enumerate(party_gender['female_pct']):
        ax.text(v + 0.3, i, f'{v:.1f}%', va='center')
    plt.tight_layout()
    plt.savefig('/app/output/03_party_gender.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Age distribution by gender
fig, ax = plt.subplots(figsize=(14, 6))
for sex, color in [('M', '#2196F3'), ('F', '#E91E63')]:
    data = recent[recent.sex == sex].age.dropna()
    ax.hist(data, bins=40, alpha=0.5, color=color, label=f'{sex} (n={len(data):,}, μ={data.mean():.1f})', edgecolor='white')
ax.set_title('Age Distribution by Gender (2011–2021)')
ax.set_xlabel('Age')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.savefig('/app/output/03_age_by_gender.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Education & Profession vs Win Rate

In [ ]:
# Win rate by education level
edu_win = recent.groupby('myneta_education').agg(
    total=('candidate', 'count'),
    wins=('is_winner', 'sum')
).reset_index()
edu_win['win_rate'] = edu_win.wins / edu_win.total * 100
edu_win = edu_win[edu_win.total >= 20].sort_values('win_rate', ascending=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Win rate
ax1.barh(edu_win.myneta_education, edu_win.win_rate, color='#7E57C2', edgecolor='white')
ax1.set_title('Win Rate by Education Level')
ax1.set_xlabel('Win Rate (%)')
for i, (_, row) in enumerate(edu_win.iterrows()):
    ax1.text(row.win_rate + 0.3, i, f'{row.win_rate:.1f}% (n={row.total})', va='center', fontsize=9)

# Count of winners
ax2.barh(edu_win.myneta_education, edu_win.wins, color='#26A69A', edgecolor='white')
ax2.set_title('Number of Winners by Education')
ax2.set_xlabel('Winners')

plt.tight_layout()
plt.savefig('/app/output/03_education_win_rate.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Profession vs win rate
prof_win = recent.groupby('tcpd_prof_main').agg(
    total=('candidate', 'count'),
    wins=('is_winner', 'sum')
).reset_index()
prof_win['win_rate'] = prof_win.wins / prof_win.total * 100
prof_win = prof_win[prof_win.total >= 20].sort_values('win_rate', ascending=True)

fig, ax = plt.subplots(figsize=(12, 6))
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(prof_win)))
ax.barh(prof_win.tcpd_prof_main, prof_win.win_rate, color=colors, edgecolor='white')
ax.set_title('Win Rate by Primary Profession (2011–2021, n≥20)')
ax.set_xlabel('Win Rate (%)')
for i, (_, row) in enumerate(prof_win.iterrows()):
    ax.text(row.win_rate + 0.3, i, f'{row.win_rate:.1f}% (n={row.total})', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('/app/output/03_profession_win_rate.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Incumbency Analysis

In [ ]:
# Incumbent win rate per election
inc_data = ge[ge.incumbent.isin(['TRUE', 'FALSE'])].copy()
inc_data['is_incumbent'] = inc_data.incumbent == 'TRUE'

inc_win = inc_data.groupby(['year', 'is_incumbent']).agg(
    total=('candidate', 'count'),
    wins=('is_winner', 'sum')
).reset_index()
inc_win['win_rate'] = inc_win.wins / inc_win.total * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

for inc, label, color in [(True, 'Incumbent', '#E53935'), (False, 'Challenger', '#1E88E5')]:
    d = inc_win[inc_win.is_incumbent == inc]
    ax1.plot(d.year, d.win_rate, 'o-', label=label, color=color, linewidth=2, markersize=8)
ax1.set_title('Win Rate: Incumbents vs Challengers')
ax1.set_xlabel('Election Year')
ax1.set_ylabel('Win Rate (%)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Incumbent survival rate (incumbents who won)
inc_only = inc_win[inc_win.is_incumbent == True]
ax2.bar(inc_only.year, inc_only.win_rate, color='#E53935', edgecolor='white', alpha=0.8)
ax2.set_title('Incumbent Survival Rate per Election')
ax2.set_xlabel('Election Year')
ax2.set_ylabel('Incumbent Win Rate (%)')
ax2.axhline(y=50, color='gray', linestyle='--', alpha=0.5)
for _, row in inc_only.iterrows():
    ax2.annotate(f'{row.win_rate:.0f}%\n({int(row.wins)}/{int(row.total)})',
                (row.year, row.win_rate), textcoords='offset points',
                xytext=(0, 5), ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('/app/output/03_incumbency.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Turncoat Analysis

In [ ]:
# Turncoat frequency and success
tc_data = ge[ge.turncoat.isin(['TRUE', 'FALSE'])].copy()
tc_data['is_turncoat'] = tc_data.turncoat == 'TRUE'

tc_by_year = tc_data.groupby(['year', 'is_turncoat']).agg(
    total=('candidate', 'count'),
    wins=('is_winner', 'sum')
).reset_index()
tc_by_year['win_rate'] = tc_by_year.wins / tc_by_year.total * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Turncoat count per election
tc_count = tc_by_year[tc_by_year.is_turncoat == True]
ax1.bar(tc_count.year, tc_count.total, color='#FF7043', edgecolor='white')
ax1.set_title('Number of Turncoats (Party Switchers) per Election')
ax1.set_xlabel('Election Year')
ax1.set_ylabel('Count')
for _, row in tc_count.iterrows():
    ax1.annotate(str(int(row.total)), (row.year, row.total),
                textcoords='offset points', xytext=(0, 5), ha='center')

# Win rate comparison
for tc, label, color in [(True, 'Turncoat', '#FF7043'), (False, 'Loyal', '#26A69A')]:
    d = tc_by_year[tc_by_year.is_turncoat == tc]
    ax2.plot(d.year, d.win_rate, 'o-', label=label, color=color, linewidth=2)
ax2.set_title('Win Rate: Turncoats vs Loyal Candidates')
ax2.set_xlabel('Election Year')
ax2.set_ylabel('Win Rate (%)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/app/output/03_turncoats.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Career Longevity

In [ ]:
# Distribution of no_terms (number of terms won)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

terms = ge.no_terms.dropna()
ax1.hist(terms, bins=range(0, int(terms.max())+2), color='#5C6BC0', edgecolor='white', align='left')
ax1.set_title('Distribution of Terms Won by Candidates')
ax1.set_xlabel('Number of Terms')
ax1.set_ylabel('Count')

# Contested count distribution
contested = ge.contested.dropna()
ax2.hist(contested, bins=range(0, int(contested.max())+2), color='#AB47BC', edgecolor='white', align='left')
ax2.set_title('Distribution of Times Contested')
ax2.set_xlabel('Times Contested')
ax2.set_ylabel('Count')

plt.tight_layout()
plt.savefig('/app/output/03_career_longevity.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Party loyalty — same_party analysis
loyalty = ge[ge.same_party.isin(['TRUE', 'FALSE'])].copy()
loyalty['is_loyal'] = loyalty.same_party == 'TRUE'

loyalty_win = loyalty.groupby(['year', 'is_loyal']).agg(
    total=('candidate', 'count'),
    wins=('is_winner', 'sum')
).reset_index()
loyalty_win['win_rate'] = loyalty_win.wins / loyalty_win.total * 100

fig, ax = plt.subplots(figsize=(14, 6))
for loyal, label, color in [(True, 'Same Party', '#26A69A'), (False, 'Changed Party', '#FF7043')]:
    d = loyalty_win[loyalty_win.is_loyal == loyal]
    ax.plot(d.year, d.win_rate, 'o-', label=label, color=color, linewidth=2)
ax.set_title('Win Rate: Party Loyal vs Party Changers')
ax.set_xlabel('Election Year')
ax.set_ylabel('Win Rate (%)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/app/output/03_party_loyalty.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top repeat winners across elections
repeat_winners = ge[ge.is_winner].groupby('candidate').agg(
    wins=('year', 'count'),
    years=('year', lambda x: sorted(x.tolist())),
    parties=('party', lambda x: list(x.unique())),
    constituencies=('constituency_name', lambda x: list(x.unique()))
).reset_index()
repeat_winners = repeat_winners[repeat_winners.wins >= 4].sort_values('wins', ascending=False)

print(f'Candidates who won 4+ general elections: {len(repeat_winners)}')
repeat_winners.head(20)